# Article-grouped XGBoost rankers

Fit one joint curator/audience LambdaMART model for root candidates and one for all comments. Query groups are article × selector, while tuning and OOF splits are made by article.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pyarrow.parquet as pq

from commentgap_analysis.ranking import run_ranker_workflow

SEED = int(os.getenv("COMMENTGAP_MODEL_SEED", "20260813"))
FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_XGB_ROOT", "model_output/selection_2025/xgboost"))
DEVICE = os.getenv("COMMENTGAP_XGB_DEVICE", "auto")
BOOTSTRAP_DRAWS = int(os.getenv("COMMENTGAP_BOOTSTRAP_DRAWS", "1000"))
feature_manifest = json.loads((FEATURE_ROOT / "feature_manifest.json").read_text())
provenance = json.loads((FEATURE_ROOT / "provenance_manifest.json").read_text())
provenance["watermark"]

## Validation design

A deterministic 15% article set is used only for light three-fold tuning across eight configurations. The remaining articles receive five-fold OOF scores. Both selector copies always stay with their article. Final full-data models are deployable artifacts, but only OOF scores are valid performance and Paper 2 evaluation inputs.

In [ ]:
results = {}
for scope in ("root", "all"):
    choice_path = FEATURE_ROOT / f"choice_set_{scope}.parquet"
    choice_set = pq.read_table(choice_path).to_pandas()
    features = list(feature_manifest["models"][scope]["features"])
    results[scope] = run_ranker_workflow(
        choice_set,
        features,
        OUTPUT_ROOT,
        scope=scope,
        device=DEVICE,
        seed=SEED,
        bootstrap_draws=BOOTSTRAP_DRAWS,
    )
results

## Artifact checks

Each scope must contain a final JSON model, five fold models, article splits, OOF scores, article-level metrics, bootstrap summaries, grouped permutation importance, TreeSHAP samples, and tie-sensitivity metrics.

In [ ]:
for scope in ("root", "all"):
    scope_root = OUTPUT_ROOT / scope
    required = [
        scope_root / "final_deployable_model.json",
        scope_root / "oof_scores_wide.parquet",
        scope_root / "metric_summary.parquet",
        scope_root / "grouped_permutation_importance.parquet",
        scope_root / "oof_treeshap_sample.parquet",
        scope_root / "tie_sensitivity_metrics.parquet",
    ]
    assert all(path.exists() for path in required), [str(path) for path in required if not path.exists()]
    assert len(list(scope_root.glob("fold_*.json"))) == 5
{scope: json.loads((OUTPUT_ROOT / scope / "model_manifest.json").read_text()) for scope in ("root", "all")}

## Next step

After the Rmd regressions and both rankers finish, run `06D_model_tables_plots.ipynb`. Do not substitute predictions from `final_deployable_model.json` for the exported article-grouped OOF scores when reporting performance.